# LangUsta: KPSS Akıl Yürütme ve Kimlik Eğitimi

## 1. Çalışma Ortamı ve Yapılandırma

In [ ]:

!pip -q install --upgrade unsloth datasets trl transformers tokenizers huggingface_hub python-dotenv

In [ ]:
# Proje yapılandırması
from pathlib import Path
import gc, hashlib, json, os, random, re, time
import unsloth
import torch
from datasets import Dataset, DatasetDict, load_dataset

RUNTIME_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
OUTPUT_ROOT = RUNTIME_ROOT / 'langusta'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
HF_OWNER = 'erhanalsr'
SOURCE_DATASET = 'AhmetSemih/Deepseek-mcq-reasoning-dataset'
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
MAIN_SYSTEM_PROMPT = 'Sen KPSS sorularını Türkçe, doğru ve gerekçeli biçimde yanıtlayan bir eğitim asistanısın.'
MAIN_DATASET_REPO = f'{HF_OWNER}/langusta-kpss-reasoning'
TOKENIZER_REPO = f'{HF_OWNER}/langusta-kpss-bpe-tokenizer'
MAIN_ADAPTER_REPO = f'{HF_OWNER}/langusta-kpss-lora'
IDENTITY_DATASET_REPO = f'{HF_OWNER}/langusta-identity'
IDENTITY_ADAPTER_REPO = f'{HF_OWNER}/langusta-identity-lora'
SEED = 3407
random.seed(SEED)

## 2. KPSS Akıl Yürütme Veri Setinin Hazırlanması

In [ ]:
# KPSS kayıtlarını ayıkla ve eğitim şemasına dönüştür
def clean_text(value):
    if value is None:
        return ''
    return ' '.join(str(value).replace('\x00', ' ').split()).strip()

def stable_id(question, answer):
    return hashlib.sha256(f'{question}\n{answer}'.encode('utf-8')).hexdigest()[:16]

def make_messages(question, answer, system_prompt):
    return [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': question},
        {'role': 'assistant', 'content': answer},
    ]

source = load_dataset(SOURCE_DATASET, split='train')
kpss_source = source.filter(lambda row: clean_text(row.get('section')).casefold() in {'kpss', 'kpss denemeleri'})
records, seen = [], set()
for row in kpss_source:
    question = clean_text(row.get('question'))
    options = [clean_text(option) for option in (row.get('options') or []) if clean_text(option)]
    if options:
        question += '\n\nSeçenekler:\n' + '\n'.join(f'{index + 1}. {option}' for index, option in enumerate(options))
    thinking = clean_text(row.get('thinking'))
    response = clean_text(row.get('response'))
    if not question or not thinking or not response:
        continue
    key = (question.casefold(), response.casefold())
    if key in seen:
        continue
    seen.add(key)
    training_answer = f'<think>\n{thinking}\n</think>\n{response}'
    records.append({
        'id': stable_id(question, response),
        'messages': make_messages(question, training_answer, MAIN_SYSTEM_PROMPT),
        'question': question, 'thinking': thinking, 'response': response,
        'section': clean_text(row.get('section')), 'topic': clean_text(row.get('topic')),
        'source': SOURCE_DATASET,
    })
records.sort(key=lambda item: item['id'])
test_size = max(2, round(len(records) * 0.10))
main_dataset = DatasetDict({
    'train': Dataset.from_list(records[test_size:]),
    'test': Dataset.from_list(records[:test_size]),
})
assert len(records) == 21, f'Beklenen 21 KPSS kaydı yerine {len(records)} bulundu.'
assert set(main_dataset['train']['id']).isdisjoint(main_dataset['test']['id'])
print(f'Ana veri: {len(main_dataset["train"])} train / {len(main_dataset["test"])} test')

In [ ]:
# Referans şemayı oluştur, eğitim dosyalarını ve tokenizer korpusunu kaydet
def to_reference_conversation(question, response, thinking=''):
    return {
        'train': [
            {'content': question, 'images': None, 'role': 'user', 'thinking': None, 'tool_calls': None},
            {'content': response, 'images': None, 'role': 'assistant', 'thinking': thinking, 'tool_calls': None},
        ]
    }
main_hub_dataset = DatasetDict({
    split_name: Dataset.from_list([to_reference_conversation(item['question'], item['response'], item['thinking']) for item in split])
    for split_name, split in main_dataset.items()
})
assert list(main_hub_dataset['train'].features) == ['train']
assert [message['role'] for message in main_hub_dataset['train'][0]['train']] == ['user', 'assistant']
main_data_dir = OUTPUT_ROOT / 'langusta-kpss-reasoning'
main_data_dir.mkdir(exist_ok=True)
for split_name, split in main_hub_dataset.items():
    split.to_json(main_data_dir / f'{split_name}.jsonl', force_ascii=False)
Dataset.from_list(records).to_json(main_data_dir / 'kpss_reasoning.jsonl', force_ascii=False)
corpus_path = main_data_dir / 'corpus.txt'
with corpus_path.open('w', encoding='utf-8') as corpus:
    for split in main_dataset.values():
        for item in split:
            corpus.write(item['messages'][1]['content'] + '\n')
            corpus.write(item['messages'][2]['content'] + '\n')

## 3. Byte-Level BPE Tokenizer Eğitimi

In [ ]:
# Tokenizerı eğit, kaydet ve Türkçe/kod örnekleriyle doğrula
from tokenizers import Tokenizer
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.trainers import BpeTrainer
from transformers import PreTrainedTokenizerFast

backend = Tokenizer(BPE(unk_token='<unk>'))
backend.pre_tokenizer = ByteLevel(add_prefix_space=False)
backend.decoder = ByteLevelDecoder()
backend.train(
    [str(corpus_path)],
    BpeTrainer(
        vocab_size=8000, min_frequency=2,
        special_tokens=['<unk>', '<pad>', '<bos>', '<eos>'],
        initial_alphabet=ByteLevel.alphabet(),
    ),
)
bpe_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=backend, unk_token='<unk>', pad_token='<pad>',
    bos_token='<bos>', eos_token='<eos>', model_max_length=2048,
)
tokenizer_dir = OUTPUT_ROOT / 'langusta-kpss-bpe-tokenizer'
bpe_tokenizer.save_pretrained(tokenizer_dir)
samples = [
    'KPSS sorularını Türkçe ve gerekçeli biçimde çözüyorum.',
    'agent = create_agent(model=model, tools=tools)',
    'İ, ı, Ş, ş, Ğ, ğ, Ü, ü, Ö, ö, Ç, ç',
]
for sample in samples:
    decoded = bpe_tokenizer.decode(bpe_tokenizer.encode(sample), skip_special_tokens=True)
    assert decoded == sample, (sample, decoded)
print(f'Tokenizer: {len(bpe_tokenizer)} token')

## 4. LangUsta Kimlik Veri Setinin Hazırlanması

In [ ]:

identity = [
('Senin adın nedir?', 'Benim adım LangUsta.'),
('Kendini tanıtır mısın?', 'Ben LangUsta, Türkçe KPSS eğitimi için geliştirilmiş bir destek asistanıyım.'),
('Seni kim oluşturdu?', 'Beni Erhan Alasar oluşturdu.'),
('Yaratıcın kim?', 'Yaratıcım Erhan Alasar’dır.'),
('Hangi amaçla geliştirildin?', 'Türkçe konuşan kullanıcılara KPSS sorularını doğru ve gerekçeli biçimde açıklamak için geliştirildim.'),
('Temel görevin nedir?', 'Temel görevim KPSS adaylarına Türkçe eğitim desteği sunmaktır.'),
('Hangi konuda uzmansın?', 'KPSS sorularını, seçenekleri ve doğru cevabın gerekçesini açıklamaya odaklanırım.'),
('Hangi dilde yardımcı olursun?', 'Öncelikle Türkçe yardımcı olurum ve teknik terimleri anlaşılır biçimde açıklarım.'),
('Bir genel amaçlı sohbet botu musun?', 'Ben özellikle KPSS eğitimi ve soru çözümleri için tasarlanmış LangUsta asistanıyım.'),
('Erhan Alasar ile bağlantın nedir?', 'Erhan Alasar benim yaratıcım ve geliştiricimdir.'),
('KPSS çalışmama yardım edebilir misin?', 'Evet. LangUsta olarak KPSS sorularını adım adım ve Türkçe açıklayabilirim.'),
('Adının anlamı nedir?', 'LangUsta adı, KPSS için Türkçe bir eğitim destek asistanı olduğumu ifade eder.'),
('Kim tarafından programlandın?', 'Erhan Alasar tarafından geliştirildim.'),
('Kullanıcılara nasıl cevap verirsin?', 'Kısa, doğru, anlaşılır ve eğitim odaklı Türkçe cevaplar vermeye çalışırım.'),
('Hangi konuları açıklıyorsun?', 'KPSS kapsamındaki soruları ve doğru cevapların gerekçelerini Türkçe açıklarım.'),
('Senin ismin ChatGPT mi?', 'Hayır. Ben Erhan Alasar tarafından geliştirilen LangUsta adlı KPSS eğitim destek asistanıyım.'),
('LangUsta ne yapar?', 'LangUsta, KPSS adaylarına Türkçe soru çözümü ve eğitim desteği sunar.'),
('Görev alanının dışında bir soru sorarsam ne yaparsın?', 'Yardımcı olabildiğim ölçüde cevaplarım; ancak asıl uzmanlık alanım KPSS eğitimidir.'),
('Varlık amacını tek cümleyle anlat.', 'Erhan Alasar tarafından, KPSS sorularını Türkçe ve anlaşılır biçimde öğretmek için oluşturuldum.'),
('Seni diğer asistanlardan ayıran özellik nedir?', 'LangUsta kimliğim ve Türkçe KPSS eğitimine özel olarak odaklanmam beni ayırır.'),
]
assert len(identity) == 20
payload = [{'question': q, 'answer': a, 'generation_method': 'manual'} for q, a in manual_identity]

In [ ]:
# Çekirdek örnekleri deterministik soru varyasyonlarıyla genişlet
question_templates = [
    '{question}',
    'Kısaca cevapla: {question}',
    'Açıkça belirt: {question}',
    'Tek cümleyle yanıtla: {question}',
    'Şu soruyu yanıtlar mısın: {question}',
]
identity_examples = []
for item in payload:
    for template_index, template in enumerate(question_templates):
        identity_examples.append({
            'question': template.format(question=item['question']),
            'answer': item['answer'],
            'generation_method': 'manual' if template_index == 0 else 'deterministic-template',
        })


identity_records = []
identity_system = 'Sen LangUsta adlı, Erhan Alasar tarafından oluşturulan Türkçe KPSS eğitim destek asistanısın. Kısa, doğru ve anlaşılır cevaplar verirsin.'
for item in identity_examples:
    identity_records.append({
        'id': stable_id(item['question'], item['answer']),
        'messages': make_messages(item['question'], item['answer'], identity_system),
        'source': item['generation_method'],
        'generation_method': item['generation_method'],
    })
identity_records.sort(key=lambda item: item['id'])
identity_test_size = max(1, round(len(identity_records) * 0.10))
identity_dataset = DatasetDict({
    'train': Dataset.from_list(identity_records[identity_test_size:]),
    'test': Dataset.from_list(identity_records[:identity_test_size]),
})
identity_data_dir = OUTPUT_ROOT / 'langusta-identity'
identity_data_dir.mkdir(exist_ok=True)
identity_hub_dataset = DatasetDict({
    split_name: Dataset.from_list([to_reference_conversation(item['messages'][1]['content'], item['messages'][2]['content']) for item in split])
    for split_name, split in identity_dataset.items()
})
assert list(identity_hub_dataset['train'].features) == ['train']
for split_name, split in identity_hub_dataset.items():
    split.to_json(identity_data_dir / f'{split_name}.jsonl', force_ascii=False)

## 5. KPSS Akıl Yürütme Uyarlaması (QLoRA)

In [ ]:

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
MAX_SEQ_LENGTH = 1024
main_model, qwen_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True,
)
main_model = FastLanguageModel.get_peft_model(
    main_model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=SEED,
)
def add_text(batch):
    return {'text': [qwen_tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch['messages']]}
main_train = main_dataset['train'].map(add_text, batched=True)
main_test = main_dataset['test'].map(add_text, batched=True)

In [ ]:

main_trainer = SFTTrainer(
    model=main_model, tokenizer=qwen_tokenizer, train_dataset=main_train, eval_dataset=main_test,
    dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, packing=False,
    args=SFTConfig(
        output_dir=str(OUTPUT_ROOT / 'main_training_checkpoints'), num_train_epochs=20,
        per_device_train_batch_size=2, gradient_accumulation_steps=4, learning_rate=2e-4,
        warmup_steps=5, logging_steps=5, eval_strategy='epoch',
        save_strategy='epoch', fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(), optim='adamw_8bit', weight_decay=0.01,
        lr_scheduler_type='linear', seed=SEED, report_to='none',
    ),
)
main_metrics = main_trainer.train().metrics
main_adapter_dir = OUTPUT_ROOT / 'langusta-kpss-lora'
main_model.save_pretrained(main_adapter_dir)
qwen_tokenizer.save_pretrained(main_adapter_dir)
print(f'Ana eğitim tamamlandı | loss={main_metrics["train_loss"]:.4f} | {main_adapter_dir}')

In [ ]:

FastLanguageModel.for_inference(main_model)
main_questions = [item['question'] for item in main_dataset['test']]
main_results = []
for question in main_questions:
    inference_messages = [
        {'role': 'system', 'content': MAIN_SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ]
    prompt = qwen_tokenizer.apply_chat_template(inference_messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_tokenizer(prompt, return_tensors='pt').to('cuda')
    output = main_model.generate(**inputs, max_new_tokens=160, do_sample=False)
    answer = qwen_tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    main_results.append({'question': question, 'answer': answer})
    print(question, '->', answer, '\n')
assert len(main_results) == len(main_dataset['test']) and all(item['answer'] for item in main_results)

In [ ]:

del main_trainer, main_model, main_train, main_test
gc.collect()
torch.cuda.empty_cache()

## 6. LangUsta Kimlik Uyarlaması (QLoRA)

In [ ]:

IDENTITY_MAX_LENGTH = 512
identity_model, identity_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=IDENTITY_MAX_LENGTH, load_in_4bit=True,
)
identity_model = FastLanguageModel.get_peft_model(
    identity_model, r=16,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=16, lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth', random_state=SEED,
)
def add_identity_text(batch):
    return {'text': [identity_tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch['messages']]}
identity_train = identity_dataset['train'].map(add_identity_text, batched=True)
identity_test = identity_dataset['test'].map(add_identity_text, batched=True)

In [ ]:

identity_trainer = SFTTrainer(
    model=identity_model, tokenizer=identity_tokenizer,
    train_dataset=identity_train, eval_dataset=identity_test,
    dataset_text_field='text', max_seq_length=IDENTITY_MAX_LENGTH, packing=False,
    args=SFTConfig(
        output_dir=str(OUTPUT_ROOT / 'identity_training_checkpoints'), num_train_epochs=12,
        per_device_train_batch_size=4, gradient_accumulation_steps=2, learning_rate=2e-4,
        warmup_steps=10, logging_steps=5, eval_strategy='epoch', save_strategy='epoch',
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='linear',
        seed=SEED, report_to='none',
    ),
)
identity_metrics = identity_trainer.train().metrics
identity_adapter_dir = OUTPUT_ROOT / 'langusta-identity-lora'
identity_model.save_pretrained(identity_adapter_dir)
identity_tokenizer.save_pretrained(identity_adapter_dir)
print(f'Kimlik eğitimi tamamlandı | loss={identity_metrics["train_loss"]:.4f} | {identity_adapter_dir}')

In [ ]:

FastLanguageModel.for_inference(identity_model)
identity_questions = ['Senin adın nedir?', 'Seni kim oluşturdu?', 'Temel görevin nedir?']
identity_results = []
for question in identity_questions:
    inference_messages = [
        {'role': 'system', 'content': identity_system},
        {'role': 'user', 'content': question},
    ]
    prompt = identity_tokenizer.apply_chat_template(inference_messages, tokenize=False, add_generation_prompt=True)
    inputs = identity_tokenizer(prompt, return_tensors='pt').to('cuda')
    output = identity_model.generate(**inputs, max_new_tokens=100, do_sample=False)
    answer = identity_tokenizer.decode(output[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
    identity_results.append({'question': question, 'answer': answer})
    print(question, '->', answer)
combined_answers = ' '.join(item['answer'] for item in identity_results)
identity_checks = {key: key in combined_answers for key in ['LangUsta', 'Erhan Alasar', 'KPSS']}
assert all(identity_checks.values()), f'Kimlik modeli kabul testini geçemedi: {identity_checks}'

## 7. Doğrulama ve Hugging Face Yayını

In [ ]:
required_paths = [
    main_data_dir / 'train.jsonl', main_data_dir / 'test.jsonl',
    tokenizer_dir / 'tokenizer.json',
    identity_data_dir / 'train.jsonl', identity_data_dir / 'test.jsonl',
    main_adapter_dir / 'adapter_config.json',
    identity_adapter_dir / 'adapter_config.json',
]
missing = [str(path) for path in required_paths if not path.exists()]
assert not missing, f'Eksik çıktılar: {missing}'
assert len(main_results) == len(main_dataset['test'])
assert len(identity_results) == 3

In [ ]:

from dotenv import load_dotenv
from huggingface_hub import HfApi

env_candidates = [RUNTIME_ROOT / '.env', RUNTIME_ROOT / 'hf.env', Path.cwd() / '.env']
env_path = next((path for path in env_candidates if path.is_file()), None)
if env_path:
    load_dotenv(env_path, override=True)
hf_token = os.getenv('HF_TOKEN', '').strip()
if not hf_token and Path('/kaggle').exists():
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN').strip()
assert hf_token, 'HF_TOKEN bulunamadı. Kaggle Secrets bölümüne HF_TOKEN ekleyin.'
api = HfApi(token=hf_token)
user = api.whoami()['name']
assert user == HF_OWNER, f'Token {user} hesabına ait; beklenen hesap: {HF_OWNER}'

main_hub_dataset.push_to_hub(MAIN_DATASET_REPO, token=hf_token, private=False)
api.upload_file(
    path_or_fileobj=main_data_dir / 'kpss_reasoning.jsonl',
    path_in_repo='source/kpss_reasoning.jsonl', repo_id=MAIN_DATASET_REPO,
    repo_type='dataset', commit_message='Add filtered KPSS source file',
)
bpe_tokenizer.push_to_hub(TOKENIZER_REPO, token=hf_token, private=False)
identity_hub_dataset.push_to_hub(IDENTITY_DATASET_REPO, token=hf_token, private=False)

for repo_id, folder in [
    (MAIN_ADAPTER_REPO, main_adapter_dir),
    (IDENTITY_ADAPTER_REPO, identity_adapter_dir),
]:
    api.create_repo(repo_id=repo_id, repo_type='model', private=False, exist_ok=True)
    api.upload_folder(
        repo_id=repo_id, repo_type='model', folder_path=str(folder),
        commit_message='Upload trained LangUsta LoRA adapter',
    )

hub_cards = {
    MAIN_DATASET_REPO: ('dataset', '''---
language: [tr]
license: apache-2.0
task_categories: [question-answering, text-generation]
pretty_name: LangUsta KPSS Reasoning
---
# LangUsta KPSS Reasoning
Contains only the `KPSS` and `KPSS Denemeleri` sections filtered from `AhmetSemih/Deepseek-mcq-reasoning-dataset`. Each assistant message preserves the original non-empty thinking trace and final response. Source license: Apache 2.0.'''),
    TOKENIZER_REPO: ('model', '''---
language: [tr]
library_name: transformers
tags: [tokenizer, bpe, kpss]
---
# LangUsta KPSS Byte-Level BPE Tokenizer
Byte-level BPE tokenizer trained on the filtered Turkish KPSS reasoning corpus with `<unk>`, `<pad>`, `<bos>`, and `<eos>` special tokens.'''),
    MAIN_ADAPTER_REPO: ('model', f'''---
base_model: {BASE_MODEL}
library_name: peft
pipeline_tag: text-generation
language: [tr]
tags: [unsloth, lora, qlora, kpss]
---
# LangUsta KPSS LoRA Adapter
PEFT adapter trained on `{MAIN_DATASET_REPO}` with Unsloth 4-bit QLoRA. This repository contains adapter weights, not a merged full model.'''),
    IDENTITY_DATASET_REPO: ('dataset', '''---
language: [tr]
license: mit
task_categories: [text-generation]
tags: [identity-finetuning, kpss, langusta]
---
# LangUsta Identity Dataset
Turkish identity dataset for LangUsta, a KPSS educational assistant created by Erhan Alasar. Twenty manually written examples are expanded to 100 records with deterministic prompt templates; no generative API is used.'''),
    IDENTITY_ADAPTER_REPO: ('model', f'''---
base_model: {BASE_MODEL}
library_name: peft
pipeline_tag: text-generation
language: [tr]
tags: [unsloth, lora, identity-finetuning, langusta]
---
# LangUsta Identity LoRA Adapter
PEFT adapter trained to identify as LangUsta, name Erhan Alasar as its creator, and describe its Turkish KPSS education role.'''),
}
for repo_id, (repo_type, card) in hub_cards.items():
    api.upload_file(
        path_or_fileobj=card.encode('utf-8'), path_in_repo='README.md',
        repo_id=repo_id, repo_type=repo_type, commit_message='Add repository card',
    )

published_urls = [
    f'https://huggingface.co/datasets/{MAIN_DATASET_REPO}',
    f'https://huggingface.co/{TOKENIZER_REPO}',
    f'https://huggingface.co/{MAIN_ADAPTER_REPO}',
    f'https://huggingface.co/datasets/{IDENTITY_DATASET_REPO}',
    f'https://huggingface.co/{IDENTITY_ADAPTER_REPO}',
]
print('Yayın tamamlandı:\n' + '\n'.join(published_urls))